# 01. Canonical adaptation and statistical EDA
After native checks in 00, map all 14 measurements and resample without interpolation.
The saved canonical development table contains the four canonical columns plus
`observed`, added by validation. It excludes the final-test period.
Detailed EDA below uses the healthy training period, so faults do not masquerade
as seasonality. No EDA coefficients are passed into the model; it fits its own
training-only references in notebook 03.

In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
from optical_anomaly.pipeline import prepare, develop, final_evaluation
from optical_anomaly.workflow import run_split

CONFIG_PATH = ROOT / "configs/config.yaml"
RUN = prepare(CONFIG_PATH)
settings = json.loads((RUN / "settings.json").read_text())
split = run_split(settings)
start = pd.Timestamp("2025-01-01", tz="UTC")
boundaries = [
    split.train_end,
    split.calibration_end,
    split.validation_end,
    split.test_end,
]
REPORT = RUN / "eda"
REPORT.mkdir(exist_ok=True)
print("Run:", RUN.resolve())

## Adapt and validate before EDA

In [ ]:
from optical_anomaly.workflow import prepare_canonical
from optical_anomaly.validation import require_downstream_rx
from optical_anomaly.diagnostics import canonical_statistics

canonical_path = prepare_canonical(RUN)
entity = pd.read_parquet(RUN / "topology.parquet").entity_id.iloc[0]
preview = pd.read_parquet(canonical_path, filters=[("entity_id", "==", entity)])
require_downstream_rx(preview)
display(preview.head(20))
print("Canonical development data:", canonical_path.resolve())

## Per-entity distributions, missingness, autocorrelation and seasonality
Read one ONT at a time to keep memory bounded. Daily and weekly lag correlations
use the regular grid: missing polls are not compressed away. Daily amplitude is
in the measurement's native canonical unit, so do not compare it across metrics.

Seasonality checks fit the first 70% of the training period and evaluate on its
last 30%. Positive holdout R² means improvement over a training-fitted constant;
negative means worse. Weekly fits require at least 21 days in that inner fitting
period. Correlation alone is not proof of seasonality. No annual seasonality claim
is possible from a 90-day dataset.

In [ ]:
from optical_anomaly.diagnostics import dependence_reports

rows, correlation_rows, autocorrelation_rows = [], [], []
for ont in pd.read_parquet(RUN / "topology.parquet").entity_id:
    training = pd.read_parquet(
        canonical_path,
        filters=[("entity_id", "==", ont), ("timestamp", "<", split.train_end)],
    )
    correlation, autocorrelation = dependence_reports(
        training, settings["generator"]["interval_minutes"]
    )
    correlation_rows.append(correlation)
    autocorrelation_rows.append(autocorrelation)
    rows.append(
        canonical_statistics(training, settings["generator"]["interval_minutes"])
    )
statistics = pd.concat(rows, ignore_index=True)
statistics.to_csv(REPORT / "canonical_training_statistics.csv", index=False)
display(statistics)
display(
    statistics.groupby("metric_name")[
        [
            "missing_fraction",
            "lag1",
            "lag_daily",
            "lag_weekly",
            "daily_holdout_r2",
            "daily_weekly_holdout_r2",
        ]
    ].median()
)
correlations = pd.concat(correlation_rows, ignore_index=True)
autocorrelations = pd.concat(autocorrelation_rows, ignore_index=True)
correlations.to_csv(REPORT / "canonical_correlations.csv", index=False)
autocorrelations.to_csv(REPORT / "canonical_autocorrelations.csv", index=False)
display(correlations)
display(autocorrelations)

## Correlation and autocorrelation alongside seasonality
Pearson measures linear association; Spearman measures monotonic association.
Reports cover every measurement pair separately per ONT, before and after removing
an intercept plus daily sine/cosine. They include paired-observation counts.
Missing readings stay on the time grid: lags do not compress gaps.

Daily residuals distinguish shared daily patterns from remaining association;
this is descriptive EDA fitted only on training, not evidence of causality.
Constant channels and insufficient pairs have undefined (NaN) coefficients.
Shared OLT measurements are not independent across ONTs. No correlation cutoff
removes features. Consider these reports with SHAP and event-level evaluation.

In [ ]:
for view in ("raw", "daily_residual"):
    pairs = correlations.loc[
        correlations.entity_id.eq(entity) & correlations.view.eq(view)
    ]
    names = sorted(set(pairs.left) | set(pairs.right))
    matrix = pd.DataFrame(np.nan, index=names, columns=names)
    for row in pairs.itertuples():
        matrix.loc[row.left, row.right] = row.spearman
        matrix.loc[row.right, row.left] = row.spearman
    fig, ax = plt.subplots(figsize=(11, 9))
    image = ax.imshow(matrix, vmin=-1, vmax=1, cmap="coolwarm")
    display_names = [
        name.replace("_fec_total_codewords", " FEC received blocks")
        .replace("_fec_corrected_codewords", " FEC corrected blocks")
        .replace("_fec_uncorrectable_codewords", " FEC uncorrectable blocks")
        .replace("_", " ")
        for name in names
    ]
    ax.set_xticks(range(len(names)), display_names, rotation=90)
    ax.set_yticks(range(len(names)), display_names)
    ax.set_title(f"{entity}: {view} Spearman correlation (diagonal omitted)")
    fig.colorbar(image, ax=ax)
    plt.tight_layout()
    plt.show()

acf = autocorrelations.loc[
    autocorrelations.entity_id.eq(entity)
    & autocorrelations.metric_name.eq("rx_power_dbm")
]
for view, group in acf.groupby("view"):
    plt.plot(group.lag_minutes / 60, group.autocorrelation, "o-", label=view)
plt.xscale("log")
plt.xlabel("Lag in hours (selected lags; logarithmic axis)")
plt.ylabel("Paired Pearson autocorrelation")
plt.title("Downstream Rx dependence before and after daily adjustment")
plt.legend()
plt.show()
print("Dependence reports:", REPORT.resolve())

## Daily profiles, distributions and cross-measurement relationships

In [ ]:
training = preview.loc[preview.timestamp < split.train_end]
wide = training.pivot(index="timestamp", columns="metric_name", values="value")
selected = [
    "rx_power_dbm",
    "upstream_rx_power_dbm",
    "ont_tx_power_dbm",
    "olt_tx_power_dbm",
    "ont_temperature_c",
    "olt_temperature_c",
]
wide[selected].hist(bins=40, figsize=(12, 7))
plt.tight_layout()
plt.show()
centred = wide[selected] - wide[selected].median()
centred.groupby(centred.index.hour).median().plot(
    subplots=True, figsize=(12, 10), title="Healthy daily profiles, one ONT"
)
plt.tight_layout()
plt.show()
display(wide[selected].corr(method="spearman"))
loss = pd.DataFrame(
    {
        "Downstream Tx minus Rx (dB)": wide.olt_tx_power_dbm - wide.rx_power_dbm,
        "Upstream Tx minus Rx (dB)": wide.ont_tx_power_dbm - wide.upstream_rx_power_dbm,
    }
)
loss.plot(figsize=(12, 3), title="Approximate optical loss relationships")
plt.show()
fec = pd.DataFrame(index=wide.index)
for direction in ("downstream", "upstream"):
    total = wide[f"{direction}_fec_total_codewords"].where(lambda x: x > 0)
    for kind in ("corrected", "uncorrectable"):
        fec[f"{direction.title()} FEC {kind} fraction"] = (
            wide[f"{direction}_fec_{kind}_codewords"] / total
        )
fec.plot(figsize=(12, 3), title="Healthy FEC fractions")
plt.show()

## Interpretation before feature engineering
Review missingness, constant sensors/counters, daily and weekly holdout results,
and relationships before proceeding. The generator has daily behaviour; a weekly
fit need not help. Temperature is optical-module context, not an automatic fault.
Shared OLT-port readings are repeated across ONTs and are not independent samples.
No automatic “realistic” verdict is produced. Record issues before fitting, and
choose a new run when changing the generator or model assumptions.